<a href="https://colab.research.google.com/github/sadhvik02/Adaptive_Hierarchical_Cyber_Attack_Detection/blob/main/Hybrid_Tech_Learning_Path_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Hybrid Tech Learning Path Recommendation System

In [56]:
# Install XGBoost if not already installed
!pip install scikit-learn xgboost lightgbm pandas matplotlib



**Prepare Dataset**

In [57]:
import pandas as pd
import random

# Seed for reproducibility
random.seed(42)

# Example pool of courses
courses = [
    "Python Basics", "Python Intermediate", "Data Science Fundamentals",
    "Machine Learning Fundamentals", "Deep Learning", "Web Development Basics",
    "JavaScript Essentials", "ReactJS Basics", "Git Basics", "Git Advanced",
    "Data Visualization with Python", "SQL Basics", "Algorithms & Data Structures",
    "AI Fundamentals", "Cloud Computing Basics"
]

# Example student interests
interests_pool = ["AI", "Data Science", "Web Development", "Cloud Computing", "Programming"]

# Simulate data for 200 students
data = []

for student_id in range(1, 201):
    # Random number of completed courses per student (2-6)
    num_completed = random.randint(2, 6)
    completed_courses = random.sample(courses, num_completed)

    # Random interest for each student
    interests = random.sample(interests_pool, 1)

    # Random scores for each completed course (50-100)
    scores = [random.randint(50, 100) for _ in completed_courses]

    # Next course (target) - pick one not yet completed
    remaining_courses = list(set(courses) - set(completed_courses))
    if remaining_courses:
        next_course = random.choice(remaining_courses)
    else:
        next_course = random.choice(courses)

    data.append({
        "student_id": student_id,
        "completed_courses": ";".join(completed_courses),
        "interests": ";".join(interests),
        "scores": ";".join(map(str, scores)),
        "next_course": next_course
    })

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv("tech_learning_path_dataset.csv", index=False)

print("Dataset created with 200 students: 'tech_learning_path_dataset.csv'")
df.head()


Dataset created with 200 students: 'tech_learning_path_dataset.csv'


,student_id,completed_courses,interests,scores,next_course
0,1,Python Basics;SQL Basics,Web Development,65;64,Python Intermediate
1,2,Data Visualization with Python;SQL Basics,Programming,55;87,AI Fundamentals
2,3,Python Basics;Python Intermediate,Data Science,64;82,Deep Learning
3,4,Git Basics;Machine Learning Fundamentals,Programming,76;64,ReactJS Basics
4,5,Deep Learning;Algorithms & Data Structures;Pyt...,Web Development,59;63;98;71;56;55,Machine Learning Fundamentals


**Preprosess Data**

In [58]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Split semicolon-separated columns into lists
df['completed_courses'] = df['completed_courses'].apply(lambda x: x.split(";"))
df['interests'] = df['interests'].apply(lambda x: x.split(";"))
df['scores'] = df['scores'].apply(lambda x: [int(i) for i in x.split(";")])

# Encode completed courses
mlb_courses = MultiLabelBinarizer()
completed_courses_encoded = mlb_courses.fit_transform(df['completed_courses'])
df_courses = pd.DataFrame(completed_courses_encoded, columns=mlb_courses.classes_)

# Encode interests
mlb_interests = MultiLabelBinarizer()
interests_encoded = mlb_interests.fit_transform(df['interests'])
df_interests = pd.DataFrame(interests_encoded, columns=mlb_interests.classes_)

# Average scores
df['avg_score'] = df['scores'].apply(lambda x: sum(x)/len(x))

# Combine features
X = pd.concat([df_courses, df_interests, df['avg_score']], axis=1)
y = df['next_course']

print("Feature set shape:", X.shape)
print("Target shape:", y.shape)


Feature set shape: (200, 21)
Target shape: (200,)


**Training ML Models**

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Label Encode the target variable
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded) # Use encoded y_train
y_pred_xgb_encoded = xgb_model.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded) # Decode predictions
print("XGBoost Accuracy:", accuracy_score(y_test_encoded, y_pred_xgb_encoded)) # Use encoded y_test
print(classification_report(y_test, y_pred_xgb)) # Use original y_test for classification_report

Random Forest Accuracy: 0.15
                                precision    recall  f1-score   support

               AI Fundamentals       0.20      0.20      0.20         5
  Algorithms & Data Structures       0.00      0.00      0.00         3
        Cloud Computing Basics       0.00      0.00      0.00         0
     Data Science Fundamentals       0.00      0.00      0.00         2
Data Visualization with Python       0.00      0.00      0.00         2
                 Deep Learning       0.00      0.00      0.00         3
                  Git Advanced       0.00      0.00      0.00         2
                    Git Basics       0.00      0.00      0.00         3
         JavaScript Essentials       0.67      0.50      0.57         4
 Machine Learning Fundamentals       0.40      0.50      0.44         4
                 Python Basics       0.00      0.00      0.00         1
           Python Intermediate       0.50      0.33      0.40         3
                ReactJS Basics    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

XGBoost Accuracy: 0.175
                                precision    recall  f1-score   support

               AI Fundamentals       0.50      0.20      0.29         5
  Algorithms & Data Structures       0.00      0.00      0.00         3
        Cloud Computing Basics       0.00      0.00      0.00         0
     Data Science Fundamentals       0.00      0.00      0.00         2
Data Visualization with Python       0.00      0.00      0.00         2
                 Deep Learning       0.00      0.00      0.00         3
                  Git Advanced       0.00      0.00      0.00         2
                    Git Basics       0.67      0.67      0.67         3
         JavaScript Essentials       0.50      0.50      0.50         4
 Machine Learning Fundamentals       0.40      0.50      0.44         4
                 Python Basics       0.00      0.00      0.00         1
           Python Intermediate       0.00      0.00      0.00         3
                ReactJS Basics       0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

TF-IDF

In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Course descriptions
courses = [
    "Python Basics", "Python Intermediate", "Data Science Fundamentals",
    "Machine Learning Fundamentals", "Deep Learning", "Web Development Basics",
    "JavaScript Essentials", "ReactJS Basics", "Git Basics", "Git Advanced",
    "Data Visualization with Python", "SQL Basics", "Algorithms & Data Structures",
    "AI Fundamentals", "Cloud Computing Basics"
]

course_descriptions = {course: course + " course covering fundamentals and advanced topics in " + " ".join(course.split()) for course in courses}

# TF-IDF vectorizer
tfidf = TfidfVectorizer()
course_vectors = tfidf.fit_transform(course_descriptions.values())

# Recommendation function
def recommend_courses_tfidf(completed_courses, top_n=3):
    completed_vectors = tfidf.transform([course_descriptions[c] for c in completed_courses])
    similarity = cosine_similarity(completed_vectors, course_vectors)
    avg_similarity = similarity.mean(axis=0)
    sorted_idx = avg_similarity.argsort()[::-1]
    recommended = []
    for idx in sorted_idx:
        course_name = list(course_descriptions.keys())[idx]
        if course_name not in completed_courses:
            recommended.append(course_name)
        if len(recommended) >= top_n:
            break
    return recommended

# Test TF-IDF
test_completed = ["Python Basics", "Git Basics"]
recommended_courses = recommend_courses_tfidf(test_completed)
print("TF-IDF Recommended Courses:", recommended_courses)


TF-IDF Recommended Courses: ['Git Advanced', 'ReactJS Basics', 'SQL Basics']


In [61]:
def explain_recommendation(student_completed, recommended_courses, student_interests):
    explanations = []
    for course in recommended_courses:
        reason = []
        for completed in student_completed:
            if completed.split()[0] in course:
                reason.append(f"Completed {completed}")
        for interest in student_interests:
            if interest.lower() in course.lower():
                reason.append(f"Interest in {interest}")
        explanations.append(f"{course} -> Reason: {', '.join(reason) if reason else 'Recommended based on similarity'}")
    return explanations

# Example explainability
explanations = explain_recommendation(test_completed, recommended_courses, ["Data Science"])
for exp in explanations:
    print(exp)


Git Advanced -> Reason: Completed Git Basics
ReactJS Basics -> Reason: Recommended based on similarity
SQL Basics -> Reason: Recommended based on similarity


In [62]:
def hybrid_recommendation(student_features, completed_courses, student_interests, top_n=3):
    """
    Hybrid recommendation:
    1. ML prediction (Random Forest / XGBoost)
    2. TF-IDF content-based recommendation
    3. Combine results with explanations
    """
    # 1️⃣ ML Prediction
    ml_pred_rf = rf_model.predict(student_features)[0]
    ml_pred_xgb = xgb_model.predict(student_features)[0]

    # Choose RF prediction for top recommendation
    top_recommendations = [ml_pred_rf] if ml_pred_rf not in completed_courses else []

    # 2️⃣ TF-IDF Recommendations
    tfidf_recs = recommend_courses_tfidf(completed_courses, top_n=top_n)

    # 3️⃣ Merge ML + TF-IDF, remove duplicates & completed courses
    for rec in tfidf_recs:
        if rec not in top_recommendations and rec not in completed_courses:
            top_recommendations.append(rec)
        if len(top_recommendations) >= top_n:
            break

    # 4️⃣ Generate explanations
    explanations = explain_recommendation(completed_courses, top_recommendations, student_interests)

    return top_recommendations, explanations


Recommedations

In [63]:
# Pick a student from dataset
student_row = df.iloc[0]
student_completed = student_row['completed_courses']
student_interests = student_row['interests']

# Prepare ML features (same as X)
student_features = pd.DataFrame([X.iloc[0]])

# Hybrid recommendation
hybrid_recs, hybrid_expl = hybrid_recommendation(student_features, student_completed, student_interests, top_n=3)

print("Hybrid Recommended Courses:", hybrid_recs)
print("\nExplanations:")
for e in hybrid_expl:
    print("-", e)


Hybrid Recommended Courses: ['Python Intermediate', 'Git Basics', 'ReactJS Basics']

Explanations:
- Python Intermediate -> Reason: Completed Python Basics
- Git Basics -> Reason: Recommended based on similarity
- ReactJS Basics -> Reason: Recommended based on similarity


**Dashbord**

In [64]:
!pip install streamlit pyngrok


##APP.py

In [65]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load pickled objects (replace with actual trained models if available)
# For demo, we will skip ML model predictions
# -----------------------------
# Dummy placeholders for ML models (optional)
# rf_model = pickle.load(open("rf_model.pkl", "rb"))

# Create TF-IDF vectorizer and course vectors
courses = [
    "Python Basics", "Python Intermediate", "Data Science Fundamentals",
    "Machine Learning Fundamentals", "Deep Learning", "Web Development Basics",
    "JavaScript Essentials", "ReactJS Basics", "Git Basics", "Git Advanced",
    "Data Visualization with Python", "SQL Basics", "Algorithms & Data Structures",
    "AI Fundamentals", "Cloud Computing Basics"
]

course_descriptions = {course: course + " course covering fundamentals and advanced topics in " + " ".join(course.split()) for course in courses}

tfidf = TfidfVectorizer()
course_vectors = tfidf.fit_transform(list(course_descriptions.values()))

# -----------------------------
# TF-IDF Recommendation
# -----------------------------
def recommend_courses_tfidf(completed_courses, top_n=3):
    completed_vectors = tfidf.transform([course_descriptions[c] for c in completed_courses])
    similarity = cosine_similarity(completed_vectors, course_vectors)
    avg_similarity = similarity.mean(axis=0)
    sorted_idx = avg_similarity.argsort()[::-1]
    recommended = []
    for idx in sorted_idx:
        course_name = list(course_descriptions.keys())[idx]
        if course_name not in completed_courses:
            recommended.append(course_name)
        if len(recommended) >= top_n:
            break
    return recommended

# -----------------------------
# Explainability
# -----------------------------
def explain_recommendation(student_completed, recommended_courses, student_interests):
    explanations = []
    for course in recommended_courses:
        reason = []
        for completed in student_completed:
            if completed.split()[0] in course:
                reason.append(f"Completed {completed}")
        for interest in student_interests:
            if interest.lower() in course.lower():
                reason.append(f"Interest in {interest}")
        explanations.append(f"{course} -> Reason: {', '.join(reason) if reason else 'Recommended based on similarity'}")
    return explanations

# -----------------------------
# Hybrid Recommendation
# -----------------------------
def hybrid_recommendation(student_features, completed_courses, student_interests, top_n=3):
    # ML prediction placeholder (demo)
    ml_pred = []  # Replace with actual RF/XGB prediction if available
    top_recommendations = [c for c in ml_pred if c not in completed_courses]

    tfidf_recs = recommend_courses_tfidf(completed_courses, top_n=top_n)
    for rec in tfidf_recs:
        if rec not in top_recommendations and rec not in completed_courses:
            top_recommendations.append(rec)
        if len(top_recommendations) >= top_n:
            break

    explanations = explain_recommendation(completed_courses, top_recommendations, student_interests)
    return top_recommendations, explanations

# -----------------------------
# Streamlit UI
# -----------------------------
st.title("Hybrid Tech Learning Path Recommendation System")

uploaded_file = st.file_uploader("Upload Student CSV", type=["csv"])
if uploaded_file:
    df_student = pd.read_csv(uploaded_file)
    st.write("Student Data:")
    st.dataframe(df_student)

    for i, row in df_student.iterrows():
        completed_courses = row['completed_courses'].split(";")
        interests = row['interests'].split(";")

        # -----------------------------
        # Generate simple ML features
        # -----------------------------
        scores = [int(s) for s in row['scores'].split(";")]
        avg_score = sum(scores)/len(scores)
        num_completed = len(completed_courses)
        student_features = pd.DataFrame([[avg_score, num_completed]], columns=['avg_score','num_completed'])

        # -----------------------------
        # Get recommendations
        # -----------------------------
        recommendations, explanations = hybrid_recommendation(student_features, completed_courses, interests)

        st.subheader(f"Student {row['student_id']} Recommendations")
        st.write(recommendations)
        st.write("Explanations:")
        for e in explanations:
            st.write("-", e)


Overwriting app.py


**Data Storage**

In [66]:
import pickle

# Save Random Forest model
with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

# Save XGBoost model
with open("xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

# Save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save TF-IDF course matrix
with open("course_matrix.pkl", "wb") as f:
    pickle.dump(course_vectors, f)


In [55]:
!pip install --upgrade pyngrok --quiet


In [67]:
!pkill ngrok


UI Token Assgining

In [69]:
# --- Setup ngrok for Streamlit in Colab ---
from pyngrok import ngrok
import os

# Kill any existing ngrok processes
!pkill ngrok

# Your ngrok auth token
NGROK_AUTH_TOKEN = "32jFbItY0j4vSoTH0SDJulKu1LA_3mCYM6TQhyx7gF8Ck267s"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Streamlit app in the background
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.headless true &")

# Create ngrok tunnel (use integer port, not string)
public_url = ngrok.connect(8501, "http")
print("🌐 Streamlit is live! Access it here:", public_url)


🌐 Streamlit is live! Access it here: NgrokTunnel: "https://83ff8fc9e140.ngrok-free.app" -> "http://localhost:8501"
